# 09 - Cross-validation of the main models

**Validation design.** The 56 training clips are split into 3 folds with `GroupKFold` by clip
(notebook 04), so all frames of a clip are in the same fold and no clip is seen in training and
evaluation at once. Run k of an experiment trains on the other two folds (every 10th frame,
about 1630 frames) and is scored on the holdout of fold k (every 10th frame, about 815 frames,
original frames without masking). The official `val` split (7 clips) is scored by every run as
an extra, fully independent check. There is no learned preprocessing: masking of ignored regions
uses the annotations of the frame itself, and normalisation is fixed by the pretrained models.

**Out-of-fold score.** The holdout predictions of the 3 runs are pooled and scored once against
all 2446 holdout frames, so every training clip is scored exactly once by a model that never saw
it. The mean and standard deviation over the folds show how much a result depends on the fold.

**What is cross-validated.** The initial and the tuned version of both models of this part
(YOLO11s: af0 at 640 px, af1 at 1152 px; RT-DETR-l: qb0 with defaults, qb3 with the paper
recipe), and the inference steps of the final pipeline on qb3. The single-change ablations of
notebooks 05-08 (NWD, top-k, queries, P2, ...) are run on fold 0 only: each RT-DETR run takes
about 3 hours on our GPU, and their effect is tested with the per-clip paired test on 26 clips.
This notebook checks that the main conclusions do not depend on fold 0.

Commands (from the repository root):

```
python scripts/prepare_data.py --fold 1 --train-step 10
python scripts/prepare_data.py --fold 2 --train-step 10
python scripts/train.py configs/query_based/qb3_bblr_nobiaswarm_1152.yaml --fold 1
python scripts/temporal.py configs/query_based/qb3_bblr_nobiaswarm_1152.yaml --strategy tiles --fold 1
```

(the same for fold 2 and for af0, af1, qb0; fold 0 is the original run of notebooks 05 and 06)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.analysis import paired_comparison, per_clip_ap
from src.common import N_FOLDS, REPO_ROOT, RUNS_DIR
from src.cross_validation import cross_validate, oof_ground_truth
from src.inference import from_coco, to_coco
from src.predict import ALL_SETS
from src.temporal import frames_table, neighbours, rescore
from src.train_tools import fold_config, load_config, run_dir

MODELS = {
    "af0 YOLO11s 640 (initial)": "configs/anchor_free/af0_baseline_640.yaml",
    "af1 YOLO11s 1152 (tuned)": "configs/anchor_free/af1_imgsz1152.yaml",
    "qb0 RT-DETR-l defaults (initial)": "configs/query_based/qb0_rtdetr_l_1152.yaml",
    "qb3 RT-DETR-l paper recipe (tuned)": "configs/query_based/qb3_bblr_nobiaswarm_1152.yaml",
}
pd.set_option("display.precision", 4)

## 1. Per-fold and out-of-fold results

In [ ]:
def fold_runs(config_path):
    # Finished runs of one experiment: fold -> run directory
    cfg = load_config(REPO_ROOT / config_path)
    found = {}
    for k in range(N_FOLDS):
        directory = run_dir(fold_config(cfg, k))
        if (directory / "metrics.json").exists():
            found[k] = directory
    return found


def holdout_predictions(runs):
    return {k: json.loads((directory / f"predictions_fold{k}.json").read_text()) for k, directory in runs.items()}


rows, predictions = [], {}
for model, config_path in MODELS.items():
    runs = fold_runs(config_path)
    if not runs:
        continue
    predictions[model] = holdout_predictions(runs)
    cv = cross_validate(predictions[model])
    val_ap = [json.loads((d / "metrics.json").read_text())["val"]["AP"] for d in runs.values()]
    row = {"model": model, "folds": len(runs)}
    row.update({f"AP_fold{k}": metrics["AP"] for k, metrics in cv["folds"].items()})
    row.update({"AP_mean": cv["mean"]["AP"], "AP_std": cv["std"]["AP"]})
    if cv["oof"]:
        row.update({f"oof_{m}": cv["oof"][m] for m in ["AP", "AP50", "AP_small", "AP_medium", "AP_large"]})
    row.update({"val_AP_mean": pd.Series(val_ap).mean(), "val_AP_std": pd.Series(val_ap).std()})
    rows.append(row)
cv_table = pd.DataFrame(rows).set_index("model")
cv_table

## 2. Do the conclusions hold in every fold?

For each comparison of notebooks 05 and 06: the AP difference in every fold, and the paired
per-clip test on all 56 training clips, each scored out of fold (19 + 19 + 18 clips; the paired
test of the earlier notebooks had 26 clips).

In [ ]:
COMPARISONS = [
    ("af1 YOLO11s 1152 (tuned)", "af0 YOLO11s 640 (initial)"),
    ("qb3 RT-DETR-l paper recipe (tuned)", "qb0 RT-DETR-l defaults (initial)"),
    ("qb0 RT-DETR-l defaults (initial)", "af1 YOLO11s 1152 (tuned)"),
    ("qb3 RT-DETR-l paper recipe (tuned)", "af1 YOLO11s 1152 (tuned)"),
]

oof_gt = oof_ground_truth()
oof_clip_ap = {model: per_clip_ap(oof_gt, [d for dets in preds.values() for d in dets])
               for model, preds in predictions.items() if len(preds) == N_FOLDS}

rows = []
for new, old in COMPARISONS:
    if new not in cv_table.index or old not in cv_table.index:
        continue
    row = {"comparison": f"{new.split()[0]} vs {old.split()[0]}"}
    for k in range(N_FOLDS):
        if f"AP_fold{k}" in cv_table:
            row[f"diff_fold{k}"] = cv_table.loc[new, f"AP_fold{k}"] - cv_table.loc[old, f"AP_fold{k}"]
    if new in oof_clip_ap and old in oof_clip_ap:
        paired = paired_comparison({old: oof_clip_ap[old], new: oof_clip_ap[new]}, reference=old).loc[new]
        row.update({"paired_diff": paired["mean_diff"], "std_err": paired["std_err"],
                    "clips_better": int(paired["clips_better"]), "clips_worse": int(paired["clips_worse"]),
                    "significant": abs(paired["mean_diff"]) >= 2 * paired["std_err"]})
    rows.append(row)
pd.DataFrame(rows).set_index("comparison")

## 3. The final pipeline out of fold

The inference steps of notebook 07 on the qb3 runs of every fold: the plain pass, full frame +
2x2 tiles, and tiles + temporal rescoring (`smooth`, radius 2, IoU 0.3, the setting chosen in
notebook 07 on fold 0 and val). Folds 1 and 2 were not used for any choice, so they are a clean
test of these choices.

In [ ]:
QB3 = "configs/query_based/qb3_bblr_nobiaswarm_1152.yaml"
TEMPORAL_SETTING = {"radius": 2, "iou_thr": 0.3, "mode": "smooth"}


def temporal_dir(fold):
    cfg = fold_config(load_config(REPO_ROOT / QB3), fold)
    return RUNS_DIR / ("temporal" if fold == 0 else "temporal_cv") / f"{cfg['name']}_tiles_{cfg['imgsz']}"


steps = {"plain": {}, "tiles": {}, "tiles + temporal": {}}
qb3_runs = fold_runs(QB3)
for k in range(N_FOLDS):
    path = temporal_dir(k) / f"detections_fold{k}.json"
    if k not in qb3_runs or not path.exists():
        continue
    split, gt_path = ALL_SETS[f"fold{k}"]
    centre_ids = [img["id"] for img in json.loads(gt_path.read_text())["images"]]
    preds = from_coco(json.loads(path.read_text()))
    window = neighbours(frames_table(split), centre_ids, TEMPORAL_SETTING["radius"])
    rescored = rescore(preds, window, **TEMPORAL_SETTING)

    steps["plain"][k] = json.loads((qb3_runs[k] / f"predictions_fold{k}.json").read_text())
    steps["tiles"][k] = [det for i in centre_ids if i in preds for det in to_coco(preds[i], i)]
    steps["tiles + temporal"][k] = [det for i, pred in rescored.items() for det in to_coco(pred, i)]

pipeline_rows, pipeline_clip_ap = [], {}
for step, by_fold in steps.items():
    if not by_fold:
        continue
    cv = cross_validate(by_fold)
    row = {"step": step, **{f"AP_fold{k}": m["AP"] for k, m in cv["folds"].items()},
           "AP_mean": cv["mean"]["AP"], "AP_std": cv["std"]["AP"]}
    if cv["oof"]:
        row.update({"oof_AP": cv["oof"]["AP"], "oof_AP50": cv["oof"]["AP50"], "oof_AP_small": cv["oof"]["AP_small"]})
        pipeline_clip_ap[step] = per_clip_ap(oof_gt, [d for dets in by_fold.values() for d in dets])
    pipeline_rows.append(row)
if pipeline_rows:
    display(pd.DataFrame(pipeline_rows).set_index("step"))

if len(pipeline_clip_ap) == 3:
    for new, old in [("tiles", "plain"), ("tiles + temporal", "tiles")]:
        comparison = paired_comparison({old: pipeline_clip_ap[old], new: pipeline_clip_ap[new]}, reference=old)
        comparison["significant"] = comparison["mean_diff"].abs() >= 2 * comparison["std_err"]
        print(f"{new} against {old} (56 clips, out of fold)")
        display(comparison.drop(index=old))

## 4. Findings

To be filled in from the results above.